In [ ]:
# ===================================================================
# Cell 1: 全局設定、導入函式庫與定義模型 Pipeline
# ===================================================================
import pandas as pd
import numpy as np
import tensorflow as tf
import lightgbm as lgb
import random
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr, pearsonr
from google.colab import drive
from tqdm import tqdm
import warnings
import re
import time
import gc
warnings.filterwarnings('ignore')

In [ ]:
# --- 1. 全局設定 ---
print("⚙️ 步驟一：全局環境設定...")
drive.mount('/content/drive', force_remount=True)
VALIDATION_DAYS = 90
TARGET_LAGS = [1, 2, 3, 4]
BASE_PATH = '/content/drive/MyDrive/'
TRAIN_PATH = os.path.join(BASE_PATH, 'mitsui-commodity-prediction-challenge/train.csv')
LABELS_PATH = os.path.join(BASE_PATH, 'mitsui-commodity-prediction-challenge/train_labels.csv')
PAIRS_PATH = os.path.join(BASE_PATH, 'mitsui-commodity-prediction-challenge/target_pairs.csv')
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
print("✅ 環境設定完成")

⚙️ 步驟一：全局環境設定...
Mounted at /content/drive
✅ 環境設定完成


In [ ]:
# --- Ensemble 設定區 ---
ENSEMBLE_METHOD = 'weighted_average'  # 'weighted_average' 或 'rank_average'
WEIGHT_A = 0
WEIGHT_B = 1

In [ ]:
# --- 2. 共享評估函數 ---
def calculate_official_sharpe_ratio(y_pred_df, y_true_df):
    if y_pred_df.shape[1] == 0: return np.nan, np.nan, np.nan
    daily_rank_correlations = []
    common_index = y_pred_df.index.intersection(y_true_df.index)
    common_cols = y_pred_df.columns.intersection(y_true_df.columns)
    pred_aligned = y_pred_df.loc[common_index, common_cols].fillna(0)
    true_aligned = y_true_df.loc[common_index, common_cols]
    for i in range(len(true_aligned)):
        true_row = true_aligned.iloc[i].dropna()
        if true_row.empty or len(true_row) < 2: continue
        pred_row = pred_aligned.iloc[i][true_row.index]
        if true_row.std() == 0 or pred_row.std() == 0: correlation = 0.0
        else: correlation, _ = spearmanr(pred_row, true_row)
        if np.isnan(correlation): correlation = 0.0
        daily_rank_correlations.append(correlation)
    daily_rank_correlations = pd.Series(daily_rank_correlations)
    if len(daily_rank_correlations) == 0 or daily_rank_correlations.std() < 1e-9: sharpe_ratio = np.nan
    else: sharpe_ratio = daily_rank_correlations.mean() / daily_rank_correlations.std()
    return sharpe_ratio, daily_rank_correlations.mean(), daily_rank_correlations.std()
print("✅ 共享評估函數定義完成")

✅ 共享評估函數定義完成


In [ ]:
##########################################################################################
#
#   模型 A (LSTM) Pipeline - 已修正訓練窗口
#
##########################################################################################
def run_pipeline_model_a(train_path, labels_path, pairs_path):
    print("\n" + "="*80 + "\n🚀 開始執行 Model A (LSTM) Pipeline...\n" + "="*80)
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    TIME_STEPS, TRAIN_DAYS, UNITS, DROPOUT_RATE, EPOCHS, BATCH_SIZE, PATIENCE, TOP_FEATURES = 30, 1827, 50, 0.2, 50, 32, 8, 5
    print("   - [Model A] 正在讀取與處理數據...")
    train_features_df = pd.read_csv(train_path)
    train_labels_df = pd.read_csv(labels_path)
    target_pairs_df = pd.read_csv(pairs_path)
    original_feature_cols = [c for c in train_features_df.columns if c != 'date_id']
    train_features_df[original_feature_cols] = train_features_df[original_feature_cols].ffill().bfill()
    df_processed = pd.merge(train_features_df, train_labels_df, on='date_id', how='left')
    price_cols = [c for c in train_features_df.columns if 'Close' in c or 'close' in c or c.startswith('FX_')]
    return_cols = [f'return_{c}' for c in price_cols]
    df_processed[return_cols] = df_processed[price_cols].pct_change()
    df_processed[return_cols] = df_processed[return_cols].rank(axis=1, pct=True)
    for col in return_cols:
        df_processed[f'ts_rank_{col}'] = df_processed[col].rolling(60, min_periods=1).rank(pct=True)
    for col in price_cols:
        delta = df_processed[col].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df_processed[f'rsi_{col}'] = 100 - (100 / (1 + (gain / loss)))
    df_processed = df_processed.drop(columns=return_cols).ffill().bfill()
    all_targets_by_lag = {lag: target_pairs_df[target_pairs_df['lag'] == lag]['target'].tolist() for lag in TARGET_LAGS}
    all_possible_targets = [c for c in df_processed.columns if c.startswith('target_')]
    base_feature_cols = [c for c in df_processed.columns if c not in all_possible_targets and c != 'date_id']
    D_max = df_processed['date_id'].max()
    val_start = D_max - VALIDATION_DAYS + 1
    train_end = val_start - 1
    train_start = train_end - TRAIN_DAYS + 1
    train_df = df_processed[(df_processed['date_id'] >= train_start) & (df_processed['date_id'] <= train_end)].copy()
    val_df = df_processed[df_processed['date_id'] >= val_start].copy()
    def find_top_features_a(df, target, features):
        corrs = df[features].corrwith(df[target]).abs().sort_values(ascending=False)
        return corrs.head(TOP_FEATURES).index.tolist()
    def create_sequences(features_df, target_series):
        X, y = [], []
        for i in range(TIME_STEPS, len(features_df)):
            X.append(features_df.iloc[i-TIME_STEPS:i].values)
            y.append(target_series.iloc[i])
        return np.array(X), np.array(y)
    def build_lstm():
        model = Sequential([LSTM(UNITS, return_sequences=True), Dropout(DROPOUT_RATE), LSTM(UNITS//2), Dropout(DROPOUT_RATE), Dense(1)])
        model.compile(optimizer='adam', loss='mse')
        return model
    all_predictions = {}
    successful_models_count = 0
    for lag in TARGET_LAGS:
        targets_list = all_targets_by_lag.get(lag, [])
        for target in tqdm(targets_list, desc=f"[Model A] Training Lag {lag}"):
            try:
                if train_df[target].notna().sum() < 50: continue
                top_features = find_top_features_a(train_df, target, base_feature_cols)
                feature_scaler, target_scaler = MinMaxScaler(), MinMaxScaler()
                train_feat_s = pd.DataFrame(feature_scaler.fit_transform(train_df[top_features]), columns=top_features, index=train_df.index)
                train_tgt_s = pd.Series(target_scaler.fit_transform(train_df[[target]]).flatten(), index=train_df.index)
                val_feat_s = pd.DataFrame(feature_scaler.transform(val_df[top_features]), columns=top_features, index=val_df.index)
                val_tgt_s = pd.Series(target_scaler.transform(val_df[[target]]).flatten(), index=val_df.index)
                X_train, y_train = create_sequences(train_feat_s, train_tgt_s)
                X_val, y_val = create_sequences(val_feat_s, val_tgt_s)
                if len(X_train) < 10 or len(X_val) == 0: continue
                model = build_lstm()
                callbacks = [EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor='val_loss', verbose=0)]
                model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), callbacks=callbacks, verbose=0)
                preds_s = model.predict(X_val, verbose=0)
                all_predictions[target] = target_scaler.inverse_transform(preds_s).flatten()
                successful_models_count += 1
            except Exception as e:
                print(f"   - [Model A] 訓練 {target} 時發生錯誤: {str(e)}")
                continue
    print(f"\n   - [Model A] 總結: 成功訓練 {successful_models_count} / {len(all_possible_targets)} 個模型。")
    y_true_df = val_df.set_index('date_id')[all_possible_targets].iloc[TIME_STEPS:]
    preds_df = pd.DataFrame(index=y_true_df.index)
    for target, preds in all_predictions.items():
        if len(preds) == len(preds_df):
            preds_df[target] = preds
    print("✅ Model A (LSTM) Pipeline 執行完畢。")
    return preds_df

In [ ]:
##########################################################################################
#
#   【修正】模型 B (LightGBM) Pipeline - 已還原 Early Stopping
#
##########################################################################################

def run_pipeline_model_b(train_path, labels_path, pairs_path):
    print("\n" + "="*80 + "\n🚀 開始執行 Model B (LightGBM) Pipeline - 完整邏輯版...\n" + "="*80)
    LGBM_PARAMS = {'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 300, 'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'num_leaves': 64, 'verbose': -1, 'n_jobs': -1, 'seed': 42}

    print("   - [Model B] 正在讀取與處理數據...")
    # (此處數據處理部分與上一版相同，為了簡潔省略，實際使用時請保留)
    train_df_raw, train_labels, target_pairs = pd.read_csv(train_path), pd.read_csv(labels_path), pd.read_csv(pairs_path)
    base_df = train_df_raw.sort_values("date_id").copy()
    ma_cols = base_df.select_dtypes(include=np.number).columns.drop('date_id', errors='ignore')
    for w in [5, 20]:
        ma = base_df[ma_cols].rolling(w, min_periods=3).mean().shift(1)
        ma.columns = [f"{c}_ma{w}" for c in ma_cols]
        base_df = pd.concat([base_df, ma], axis=1)
    lag_cols = [c for c in train_df_raw.columns if c != 'date_id']
    for lag in [1, 2, 3, 7]:
        lags = base_df[lag_cols].shift(lag)
        lags.columns = [f"{c}_lag_{lag}" for c in lag_cols]
        base_df = pd.concat([base_df, lags], axis=1)
    log_cols = [c for c in base_df.columns if c != 'date_id' and pd.api.types.is_numeric_dtype(base_df[c])]
    base_df[log_cols] = base_df[log_cols].clip(lower=-1.0 + 1e-9).apply(np.log1p)

    D_max = base_df['date_id'].max()
    val_start_date_id = D_max - VALIDATION_DAYS + 1
    train_df = base_df[base_df['date_id'] < val_start_date_id].copy()
    validation_df = base_df[base_df['date_id'] >= val_start_date_id].copy()

    print("   - [Model B] 執行複雜特徵篩選...")
    # (此處特徵篩選輔助函數與上一版相同，為了簡潔省略，實際使用時請保留)
    seed_cols = target_pairs['pair'].str.split('-').explode().str.strip().dropna().unique().tolist()
    seed2feats = select_features_from_seeds(df=train_df, seed_cols=seed_cols)
    seed_groups = assign_features_to_primary_seed(df=train_df, seed_cols=seed_cols, seed2feats=seed2feats)
    target_to_seeds = read_target_to_seeds(target_pairs)
    target2features = build_target_feature_subsets(df=train_df, target_to_seeds=target_to_seeds, seed_groups=seed_groups)
    target2weights = compute_feature_weights(df=train_df, y=train_labels, target2features=target2features)

    X_train_full = train_df.set_index('date_id')
    X_val_full = validation_df.set_index('date_id')
    y_train_true = train_labels[train_labels['date_id'].isin(X_train_full.index)].set_index('date_id')
    y_validation_true = train_labels[train_labels['date_id'].isin(X_val_full.index)].set_index('date_id')

    all_predictions = {}
    successful_models_count = 0
    for target in tqdm(target_pairs['target'], desc="[Model B] Training all targets"):
        try:
            feats = [f for f in target2features.get(target, []) if f in X_train_full.columns]
            if not feats or target not in y_train_true.columns: continue

            y_tr = y_train_true[target].dropna()
            if len(y_tr) < 50: continue

            weights = target2weights.get(target, {f: 1.0 for f in feats})
            weight_series = pd.Series(weights, index=feats).reindex(feats, fill_value=1.0)

            X_tr = X_train_full.loc[y_tr.index, feats].mul(weight_series, axis=1)

            # --- 【修正】開始 ---
            # 準備 Early Stopping 需要的驗證集
            eval_set = [(X_tr, y_tr)]
            callbacks = []

            if target in y_validation_true.columns and y_validation_true[target].notna().any():
                y_val = y_validation_true[target].dropna()
                common_ids = X_val_full.index.intersection(y_val.index)
                if len(common_ids) > 0:
                    X_val_es = X_val_full.loc[common_ids, feats].mul(weight_series, axis=1)
                    y_val_es = y_val.loc[common_ids]
                    eval_set.append((X_val_es, y_val_es))
                    callbacks.append(lgb.early_stopping(stopping_rounds=50, verbose=False))

            model = lgb.LGBMRegressor(**LGBM_PARAMS)

            # 將 eval_set 和 callbacks 傳遞給 fit 方法
            model.fit(X_tr, y_tr,
                      eval_set=eval_set,
                      callbacks=callbacks)
            # --- 【修正】結束 ---

            X_val_weighted = X_val_full[feats].mul(weight_series, axis=1)
            all_predictions[target] = model.predict(X_val_weighted)
            successful_models_count += 1
        except Exception as e:
            print(f"   - [Model B] 訓練 {target} 時發生錯誤: {str(e)}")
            continue

    print(f"\n   - [Model B] 總結: 成功訓練 {successful_models_count} / {len(target_pairs)} 個模型。")
    preds_df = pd.DataFrame(index=X_val_full.index)
    for target, preds in all_predictions.items():
        preds_df[target] = pd.Series(preds, index=X_val_full.index)

    print("✅ Model B (LightGBM) Pipeline 執行完畢。")
    return preds_df

In [ ]:
# ===================================================================
# Part 1: 訓練 Model A 並儲存結果
# ===================================================================
print("\n--- 開始執行第一部分：訓練 Model A ---")

# 執行 Model A 的完整 Pipeline
predictions_a = run_pipeline_model_a(TRAIN_PATH, LABELS_PATH, PAIRS_PATH)

# 建立儲存路徑
PRED_A_PATH = os.path.join(BASE_PATH, 'predictions_model_a.csv')

# 將 Model A 的預測結果儲存到 Google Drive
predictions_a.to_csv(PRED_A_PATH)
print(f"\n✅ Model A 預測結果已成功儲存至: {PRED_A_PATH}")

# 清理記憶體，為下個模型做準備
del predictions_a
gc.collect()

print("\n--- 第一部分執行完畢 ---")


--- 開始執行第一部分：訓練 Model A ---

🚀 開始執行 Model A (LSTM) Pipeline...
   - [Model A] 正在讀取與處理數據...


[Model A] Training Lag 4: 100%|██████████| 106/106 [24:07<00:00, 13.65s/it]



   - [Model A] 總結: 成功訓練 424 / 424 個模型。
✅ Model A (LSTM) Pipeline 執行完畢。

✅ Model A 預測結果已成功儲存至: /content/drive/MyDrive/predictions_model_a.csv

--- 第一部分執行完畢 ---


In [ ]:
# ===================================================================
# Part 2: 訓練 Model B 並儲存結果
# ===================================================================
print("\n--- 開始執行第二部分：訓練 Model B ---")

# 執行 Model B 的完整 Pipeline
predictions_b = run_pipeline_model_b(TRAIN_PATH, LABELS_PATH, PAIRS_PATH)

# 建立儲存路徑
PRED_B_PATH = os.path.join(BASE_PATH, 'predictions_model_b.csv')

# 將 Model B 的預測結果儲存到 Google Drive
predictions_b.to_csv(PRED_B_PATH)
print(f"\n✅ Model B 預測結果已成功儲存至: {PRED_B_PATH}")

# 清理記憶體
del predictions_b
gc.collect()

print("\n--- 第二部分執行完畢 ---")


--- 開始執行第二部分：訓練 Model B ---

🚀 開始執行 Model B (LightGBM) Pipeline - 完整邏輯版...
   - [Model B] 正在讀取與處理數據...
   - [Model B] 執行複雜特徵篩選...


[Model B] Training all targets: 100%|██████████| 424/424 [00:42<00:00,  9.87it/s]



   - [Model B] 總結: 成功訓練 424 / 424 個模型。
✅ Model B (LightGBM) Pipeline 執行完畢。

✅ Model B 預測結果已成功儲存至: /content/drive/MyDrive/predictions_model_b.csv

--- 第二部分執行完畢 ---


In [ ]:
# ===================================================================
# Part 3: 載入結果、Ensemble 並進行最終評估
# ===================================================================
print("\n--- 開始執行第三部分：融合與評估 ---")

# --- 從 Google Drive 載入已儲存的預測結果 ---
PRED_A_PATH = os.path.join(BASE_PATH, 'predictions_model_a.csv')
PRED_B_PATH = os.path.join(BASE_PATH, 'predictions_model_b.csv')

print("   - 正在從硬碟載入 Model A 的預測...")
predictions_a = pd.read_csv(PRED_A_PATH, index_col='date_id')
print("   - 正在從硬碟載入 Model B 的預測...")
predictions_b = pd.read_csv(PRED_B_PATH, index_col='date_id')
print("✅ 預測結果載入成功！")

# --- 執行融合策略 ---
print("\n" + "="*80)
print("🤝 步驟四：融合 (Ensemble) 預測結果...")
print("="*80)

common_cols = predictions_a.columns.intersection(predictions_b.columns)
common_index = predictions_a.index.intersection(predictions_b.index)
preds_a_aligned = predictions_a.loc[common_index, common_cols]
preds_b_aligned = predictions_b.loc[common_index, common_cols]

ensembled_predictions = pd.DataFrame(index=common_index)

if ENSEMBLE_METHOD == 'weighted_average':
    print(f"   - 執行策略: 加權平均法 (Weighted Average)")
    print(f"   - 使用權重: Model A = {WEIGHT_A}, Model B = {WEIGHT_B}")
    ensembled_predictions = (WEIGHT_A * preds_a_aligned) + (WEIGHT_B * preds_b_aligned)
elif ENSEMBLE_METHOD == 'rank_average':
    print(f"   - 執行策略: 排名平均法 (Rank Average)")
    print(f"   - 使用權重: Model A = {WEIGHT_A}, Model B = {WEIGHT_B}")
    ranks_a = preds_a_aligned.rank(axis=1, method='average')
    ranks_b = preds_b_aligned.rank(axis=1, method='average')
    ensembled_predictions = (WEIGHT_A * ranks_a) + (WEIGHT_B * ranks_b)
else:
    print(f"   - 錯誤: 未知的 ENSEMBLE_METHOD '{ENSEMBLE_METHOD}'。將預設使用加權平均法。")
    ensembled_predictions = (WEIGHT_A * preds_a_aligned) + (WEIGHT_B * preds_b_aligned)

print(f"✅ 預測融合完畢，維度: {ensembled_predictions.shape}")

# --- 執行最終評估 ---
print("\n" + "="*80 + "\n🏆 步驟五：最終綜合評估\n" + "="*80)

all_possible_targets = [f'target_{i}' for i in range(424)]
true_values_df = pd.read_csv(LABELS_PATH)
val_dates = true_values_df['date_id'].unique()[-VALIDATION_DAYS:]
y_true_final = true_values_df[true_values_df['date_id'].isin(val_dates)].set_index('date_id')

sharpe, mu, sigma = calculate_official_sharpe_ratio(ensembled_predictions, y_true_final)

print(f"\n--- 最終綜合評估結果 (Ensemble Method: {ENSEMBLE_METHOD}) ---")
print(f"   📊 平均每日排序相關性 (μ): {mu:.6f}")
print(f"   📊 排序相關性標準差 (σ): {sigma:.6f}")
print(f"   🏆 計算出的綜合官方夏普比率: {sharpe:.4f}")

if sharpe > 2.0: performance = "🏆 優秀"
elif sharpe > 1.0: performance = "🥇 良好"
elif sharpe > 0.5: performance = "🥈 一般"
else: performance = "🥉 需改進"
print(f"   📈 綜合模型表現: {performance}")
print("\n--- 第三部分執行完畢！ ---")


--- 開始執行第三部分：融合與評估 ---
   - 正在從硬碟載入 Model A 的預測...
   - 正在從硬碟載入 Model B 的預測...
✅ 預測結果載入成功！

🤝 步驟四：融合 (Ensemble) 預測結果...
   - 執行策略: 加權平均法 (Weighted Average)
   - 使用權重: Model A = 0, Model B = 1
✅ 預測融合完畢，維度: (60, 424)

🏆 步驟五：最終綜合評估

--- 最終綜合評估結果 (Ensemble Method: weighted_average) ---
   📊 平均每日排序相關性 (μ): 0.129224
   📊 排序相關性標準差 (σ): 0.130365
   🏆 計算出的綜合官方夏普比率: 0.9912
   📈 綜合模型表現: 🥈 一般

--- 第三部分執行完畢！ ---
